# Importação inicial do dataset

In [20]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, TensorDataset

path_dataset = "/content/drive/MyDrive/Deep Learning/ed2022-stata.dta"

df = pd.read_stata(path_dataset, convert_categoricals=False)

col_triage = [
    "IMMEDR",  # Target (Nível de Urgência / ESI 1-5)
    "RFV1",  # Queixa principal
    "AGE",
    "SEX",  # Variáveis demográficas
    "TEMPF",  # Temperatura em °F
    "PULSE",  # Frequência Cardíaca
    "RESPR",  # Frequência Respiratória
    "BPSYS",  # Pressão Sistólica
    "BPDIAS",  # Pressão Diastólica
    "PAINSCALE",  # Escala de Dor (0-10)
    "POPCT",  # Saturação de Oxigênio (% SpO2)
]

col_selected = [c for c in col_triage if c in df.columns]
df_triage = df[col_selected].copy()

# Converter Temperatura de °F para °C
df_triage["TEMPF"] = ((df_triage["TEMPF"]) / 10.0 - 32) * 5 / 9

# Substituir códigos de dados ausentes padrão do NHAMCS por NaN
df_triage = df_triage.replace([-9, -8, -7, -1, 999, 998], np.nan)

# Garantir filtragem do Target válido (IMMEDR entre 1 e 5)
df_triage = df_triage[df_triage["IMMEDR"].isin([1, 2, 3, 4, 5])].copy()

# Mapear Target para classes de 0 a 4 (Padrão PyTorch CrossEntropyLoss)
df_triage["target"] = df_triage["IMMEDR"].astype(int) - 1

# Padronização e normalização dos dados

In [21]:
vital_signs = [
    "TEMPF",
    "PULSE",
    "RESPR",
    "BPSYS",
    "BPDIAS",
    "PAINSCALE",
    "POPCT",
]

# A) Criar máscaras binárias indicadoras de ausência (Missing Value Indicators)
for col in vital_signs:
    if col in df_triage.columns:
        df_triage[f"{col}_is_missing"] = df_triage[col].isna().astype(int)

# B) Imputação fisiológica por grupo etário
conditions = [
    (df_triage["AGE"] < 2),
    (df_triage["AGE"] >= 2) & (df_triage["AGE"] < 12),
    (df_triage["AGE"] >= 12) & (df_triage["AGE"] < 65),
    (df_triage["AGE"] >= 65),
]
choices = ["lactente", "pediatrico", "adulto", "idoso"]
df_triage["age_group"] = np.select(conditions, choices, default="adulto")

for col in vital_signs:
    if col in df_triage.columns:
        df_triage[col] = df_triage.groupby("age_group")[col].transform(
            lambda x: x.fillna(x.median())
        )

df_triage.drop(columns=["age_group"], inplace=True)

# C) Codificação de Variáveis Categóricas
df_triage["SEX"] = df_triage["SEX"].map({1: 0, 2: 1}).fillna(0)

# Top 30 queixas principais (RFV1) via One-Hot Encoding
top_rfv = df_triage["RFV1"].value_counts().nlargest(30).index
df_triage["RFV1_top"] = df_triage["RFV1"].apply(
    lambda x: x if x in top_rfv else -1
)
df_triage = pd.get_dummies(
    df_triage, columns=["RFV1_top"], prefix="rfv", drop_first=True
)
df_triage.drop(columns=["RFV1"], inplace=True)

# Divisão em X (features) e y (target)
X = df_triage.drop(columns=["IMMEDR", "target"])
y = df_triage["target"].values

# CONVERSÃO CRÍTICA: Garante que booleanos e inteiros sejam convertidos para float32
X = X.astype(np.float32)

# Divisão Estratificada (Treino e Teste)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Padronização de escala das variáveis contínuas
num_cols = ["AGE"] + vital_signs
scaler = StandardScaler()
X_train[num_cols] = scaler.fit_transform(X_train[num_cols])
X_test[num_cols] = scaler.transform(X_test[num_cols])

# Converte para Tensores PyTorch (Agora sem o erro de dtype object)
X_train_tensor = torch.tensor(X_train.values, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.long)
X_test_tensor = torch.tensor(X_test.values, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.long)

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

# Rede neural - PyTorch

In [40]:
class ESITriageClassifier(nn.Module):

    def __init__(self, input_dim, num_classes=5):
        super(ESITriageClassifier, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, num_classes),
        )

    def forward(self, x):
        return self.network(x)


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
input_dim = X_train.shape[1]
model = ESITriageClassifier(input_dim=input_dim, num_classes=5).to(device)

# Cálculo dos pesos de classe para lidar com o desbalanceamento das emergências
class_counts = np.bincount(y_train)
#class_weights = 1.0 / np.sqrt(class_counts)
class_weights = 1.0 / class_counts**0.25
class_weights = class_weights / class_weights.sum()
weights_tensor = torch.tensor(class_weights, dtype=torch.float32).to(device)

criterion = nn.CrossEntropyLoss(weight=weights_tensor)
optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)

# Treinamento do modelo

In [41]:
epochs = 20

for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * inputs.size(0)
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

    epoch_loss = running_loss / total
    epoch_acc = (correct / total) * 100

    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(
            f"Época [{epoch+1}/{epochs}] - Perda (Loss): {epoch_loss:.4f} | Acurácia de Treino: {epoch_acc:.2f}%"
        )

Época [1/20] - Perda (Loss): 1.3190 | Acurácia de Treino: 47.19%
Época [5/20] - Perda (Loss): 1.1267 | Acurácia de Treino: 56.24%
Época [10/20] - Perda (Loss): 1.0942 | Acurácia de Treino: 56.74%
Época [15/20] - Perda (Loss): 1.0784 | Acurácia de Treino: 57.40%
Época [20/20] - Perda (Loss): 1.0644 | Acurácia de Treino: 57.80%


# Avaliação do modelo

In [42]:
model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for inputs, labels in test_loader:
        inputs = inputs.to(device)
        outputs = model(inputs)
        _, predicted = outputs.max(1)

        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.numpy())

# Relatório de Métricas Clínicas
esi_labels = [
    "ESI 1 (Emergência)",
    "ESI 2 (Muito Urgente)",
    "ESI 3 (Urgente)",
    "ESI 4 (Pouco Urgente)",
    "ESI 5 (Não Urgente)",
]

print("\n--- Relatório de Classificação (ESI) ---")
print(classification_report(
    all_labels,
    all_preds,
    target_names=esi_labels
))

print("\n--- Matriz de Confusão ---")
print(confusion_matrix(all_labels, all_preds))


# ============================================================
# MÉTRICAS ORDINAIS DO ESI
# ============================================================

# O modelo utiliza as classes 0, 1, 2, 3, 4.
# Portanto, a distância entre as classes representa
# a distância entre os níveis ESI.
erros = np.abs(
    np.array(all_labels) - np.array(all_preds)
)

# Acurácia exata
acuracia_exata = np.mean(erros == 0) * 100

# Erro absoluto médio
mae_esi = np.mean(erros)

# Previsões com erro de no máximo 1 nível
erro_ate_1 = np.mean(erros <= 1) * 100

# Erros de 2 ou mais níveis
erro_2_ou_mais = np.mean(erros >= 2) * 100

print("\n--- Métricas Ordinais do ESI ---")
print(f"Acurácia exata:                {acuracia_exata:.2f}%")
print(f"MAE (erro médio):              {mae_esi:.3f} níveis")
print(f"Previsões com erro <= 1 nível: {erro_ate_1:.2f}%")
print(f"Erros de >= 2 níveis:          {erro_2_ou_mais:.2f}%")

# Distribuição dos erros
print("\n--- Distribuição da Distância entre ESI Real e Previsto ---")

for distancia in range(5):
    quantidade = np.sum(erros == distancia)
    percentual = quantidade / len(erros) * 100

    print(
        f"Erro de {distancia} nível(is): "
        f"{quantidade:4d} pacientes ({percentual:.2f}%)"
    )


# Salvar os pesos do modelo e o scaler para produção
torch.save(model.state_dict(), "modelo_triagem_esi.pth")

import joblib
joblib.dump(scaler, "scaler_triagem.pkl")

print(
    "\n✔ Treinamento concluído com sucesso! "
    "Pesos salvos em 'modelo_triagem_esi.pth'."
)


--- Relatório de Classificação (ESI) ---
                       precision    recall  f1-score   support

   ESI 1 (Emergência)       0.25      0.07      0.11        28
ESI 2 (Muito Urgente)       0.44      0.24      0.31       319
      ESI 3 (Urgente)       0.60      0.70      0.65      1068
ESI 4 (Pouco Urgente)       0.49      0.53      0.51       565
  ESI 5 (Não Urgente)       0.00      0.00      0.00        62

             accuracy                           0.55      2042
            macro avg       0.36      0.31      0.32      2042
         weighted avg       0.52      0.55      0.53      2042


--- Matriz de Confusão ---
[[  2   7  16   3   0]
 [  5  75 203  36   0]
 [  1  76 750 241   0]
 [  0   9 254 302   0]
 [  0   3  27  32   0]]

--- Métricas Ordinais do ESI ---
Acurácia exata:                55.29%
MAE (erro médio):              0.497 níveis
Previsões com erro <= 1 nível: 95.35%
Erros de >= 2 níveis:          4.65%

--- Distribuição da Distância entre ESI Real e Previ

/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
